# Session 3: a loaded generator, transformer losses, and saturation
**A. Real-world problem.** A bicycle dynamo lights a 3 W lamp. The coil has resistance and
inductance, so the lamp never gets the full EMF. Real transformers run warm because their
wire has resistance and their iron wastes energy. And if you push too much flux into the
iron, the output waveform is no longer a clean sine.

**Plan.** (1) solve the generator circuit in time (RK4) and compare with the exact steady
state, (2) check energy conservation and Lenz's law, (3) a transformer with losses, (4) a
core that saturates, (5) the mini-project: size a dynamo.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the generator circuit
A coil with armature resistance Ra and inductance L drives a resistive load RL. The coil EMF is
e(t) = k w sin(w t) with k = N B A. Kirchhoff's voltage law:  L di/dt = e - (Ra + RL) i.
We integrate it with the classical 4th-order Runge-Kutta method (RK4). We also integrate
three energies at the same time: into the load, into the coil resistance, and the mechanical
work needed to turn the coil. The torque of the magnetic field on the coil is
tau = -k i sin(w t) (negative = it opposes the rotation, which is Lenz's law), so the
mechanical power we must supply is -tau * w = k w i sin(w t) = e i.

Exact steady state (hand calculation): impedance Z = sqrt((Ra + RL)^2 + (w L)^2), peak current
Em / Z, RMS current Em / (Z sqrt 2), load power Irms^2 RL.

In [ ]:
def steady_state(turns, b_field, area, freq, ra, rl, ind):
    """Exact sinusoidal steady state of the generator circuit."""
    w = 2 * np.pi * freq
    em = turns * b_field * area * w
    z = float(np.hypot(ra + rl, w * ind))
    irms = em / z / np.sqrt(2.0)
    return {"em": em, "z": z, "i_peak": em / z, "i_rms": irms, "p_load": irms ** 2 * rl,
            "p_arm": irms ** 2 * ra, "phase_deg": float(np.degrees(np.arctan2(w * ind, ra + rl))),
            "v_load_rms": irms * rl}


def simulate_generator(turns, b_field, area, freq, ra, rl, ind, periods=24, steps=400, window=4):
    """RK4 time simulation. Returns a dict with time, current, torque and measured results.

    Measurements use the last `window` whole periods, after the start-up transient has died away.
    """
    w = 2 * np.pi * freq
    k = turns * b_field * area
    h = 1.0 / freq / steps
    n = periods * steps

    def f(t, y):
        s = np.sin(w * t)
        i = y[0]
        e = k * w * s
        return np.array([(e - (ra + rl) * i) / ind,      # coil current
                         rl * i * i,                      # energy into the load
                         ra * i * i,                      # energy into the coil resistance
                         k * w * s * i])                  # mechanical work: -torque * w

    y = np.zeros(4)
    ts = np.arange(n + 1) * h
    ys = np.zeros((n + 1, 4))
    for j in range(n):
        t = ts[j]
        k1 = f(t, y)
        k2 = f(t + h / 2, y + h / 2 * k1)
        k3 = f(t + h / 2, y + h / 2 * k2)
        k4 = f(t + h, y + h * k3)
        y = y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        ys[j + 1] = y
    torque = -k * ys[:, 0] * np.sin(w * ts)          # magnetic torque on the coil (N m)
    j0 = n - window * steps
    span = ts[n] - ts[j0]
    return {"t": ts, "i": ys[:, 0], "torque": torque, "omega": w,
            "i_rms": float(np.sqrt((ys[n, 1] - ys[j0, 1]) / rl / span)),
            "p_load": float((ys[n, 1] - ys[j0, 1]) / span),
            "p_arm": float((ys[n, 2] - ys[j0, 2]) / span),
            "p_mech": float((ys[n, 3] - ys[j0, 3]) / span),
            "stored": float(0.5 * ind * ys[n, 0] ** 2 - 0.5 * ind * ys[j0, 0] ** 2),
            "w_mech": float(ys[n, 3] - ys[j0, 3]), "e_load": float(ys[n, 1] - ys[j0, 1]),
            "e_arm": float(ys[n, 2] - ys[j0, 2]), "span": float(span)}


def energy_balance(sim, ind=None):
    """Mechanical work in minus (load energy + coil heat + change in stored magnetic energy), in joules."""
    return sim["w_mech"] - sim["e_load"] - sim["e_arm"] - sim["stored"]

## Step 2: a transformer with losses (illustrative numbers)
Real transformer, referred to the primary side. Turns ratio a = N1/N2.
* copper loss: the winding resistances, Req = R1 + a^2 R2, carry the load current;
* leakage: a series reactance Xeq (flux that links only one winding) causes a voltage drop;
* core loss: a resistor Rc across the primary, P_core = V1^2 / Rc, present even with no load.
Efficiency = P_out / P_in. Voltage regulation = (V2 no load - V2 full load) / V2 no load.
Values below are illustrative for a small 230 V : 12 V transformer.

In [ ]:
XFMR = {"n1": 460, "n2": 24, "r1": 30.0, "r2": 0.08, "x_eq": 40.0, "rc": 35000.0}


def transformer_load(r_load, v1=230.0, p=None):
    """Phasor solution of the equivalent circuit with a resistive load r_load (ohm) on the secondary."""
    p = XFMR if p is None else p
    a = p["n1"] / p["n2"]
    req = p["r1"] + a * a * p["r2"]
    zs = complex(req, p["x_eq"])
    rl_ref = a * a * r_load
    i_ref = v1 / (zs + rl_ref)                       # load current referred to the primary (A)
    i_core = v1 / p["rc"]
    p_out = abs(i_ref) ** 2 * rl_ref
    p_cu = abs(i_ref) ** 2 * req
    p_core = v1 ** 2 / p["rc"]
    p_in = (v1 * np.conj(i_ref + i_core)).real
    v2 = abs(i_ref) * rl_ref / a
    return {"v2": float(v2), "i2": float(abs(i_ref) * a), "i1": float(abs(i_ref + i_core)),
            "p_out": float(p_out), "p_cu": float(p_cu), "p_core": float(p_core), "p_in": float(p_in),
            "eff": float(p_out / p_in), "v2_noload": float(v1 / a),
            "regulation": float((v1 / a - v2) / (v1 / a))}

## Step 3: a core that saturates (the failure case, illustrative)
Iron cannot carry unlimited flux. We model the winding flux linkage as
lambda = lambda_s tanh(i / i_s): proportional to current for small currents, but never above
lambda_s. A source of peak voltage vp with series resistance rs gives d(lambda)/dt = v - rs i.
This equation is *stiff* near saturation (the incremental inductance collapses), so we use
the implicit backward-Euler method, solving for i with bisection at every step. The voltage
on the secondary (open circuit) is d(lambda)/dt / a. This is a teaching model with made-up
numbers, not a model of a real core.

In [ ]:
def saturating_core(drive, freq=50.0, v_rated_peak=12 * np.sqrt(2), lam_s=0.108, i_s=0.05, rs=4.0,
                    periods=8, steps=2000):
    """Drive the saturating core with `drive` times the rated voltage. Returns time, current, winding voltage."""
    w = 2 * np.pi * freq
    h = 1.0 / freq / steps
    vp = drive * v_rated_peak
    n = periods * steps
    ts = np.arange(n + 1) * h
    cur = np.zeros(n + 1)
    lam = 0.0
    for j in range(1, n + 1):
        v = vp * np.cos(w * ts[j])
        lo, hi = -50.0, 50.0                          # residual g(i) = lam_s tanh(i/i_s) - lam - h (v - rs i)
        for _ in range(60):
            mid = 0.5 * (lo + hi)
            g = lam_s * np.tanh(mid / i_s) - lam - h * (v - rs * mid)
            if g > 0:
                hi = mid
            else:
                lo = mid
        cur[j] = 0.5 * (lo + hi)
        lam = lam_s * np.tanh(cur[j] / i_s)
    v_wind = vp * np.cos(w * ts) - rs * cur
    return {"t": ts, "i": cur, "v_wind": v_wind, "steps": steps, "periods": periods}


def thd(wave, steps, periods, skip=2):
    """Total harmonic distortion of a waveform: sqrt(sum of harmonics 2.. squared) / fundamental.

    Uses whole periods after the first `skip` periods.
    """
    x = np.asarray(wave[skip * steps: periods * steps], dtype=float)
    spec = np.abs(np.fft.rfft(x))
    m = periods - skip                                 # fundamental sits in bin m
    fund = spec[m]
    harm = np.sqrt(np.sum(spec[2 * m::m][:20] ** 2))
    return float(harm / fund)

## Step 4: mini-project - a bicycle dynamo for a 3 W lamp
A roller of radius 0.01 m runs on the tyre; the bike speed v gives w = v / 0.01 (rad/s) and the
electrical frequency f = w / (2 pi). The lamp is 6 V, 3 W: RL = 6^2 / 3 = 12 ohm. The coil has
B = 0.3 T, A = 4e-4 m^2, Ra = 1.5 ohm and L = 40 mH (illustrative; in reality L grows with N^2,
which this simple model ignores).

In [ ]:
DYNAMO = {"b": 0.3, "area": 4e-4, "ra": 1.5, "ind": 0.040, "rl": 12.0, "roller": 0.01}


def dynamo_power(turns, speed_kmh, d=None):
    """Lamp power (W) at a bike speed (km/h), from the exact steady state."""
    d = DYNAMO if d is None else d
    f = (speed_kmh / 3.6) / d["roller"] / (2 * np.pi)
    return steady_state(turns, d["b"], d["area"], f, d["ra"], d["rl"], d["ind"])["p_load"]

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    GEN = dict(turns=250, b_field=0.2, area=0.0016, freq=50.0, ra=2.0, rl=10.0, ind=0.03)
    ss = steady_state(**GEN)
    sim = simulate_generator(**GEN)
    print(f"generator: peak EMF {ss['em']:.2f} V, Z {ss['z']:.2f} ohm, phase lag {ss['phase_deg']:.1f} deg")
    print(f"  steady-state I_rms {ss['i_rms']:.4f} A, load power {ss['p_load']:.3f} W")
    print(f"  RK4 measured  I_rms {sim['i_rms']:.4f} A, load power {sim['p_load']:.3f} W, error {abs(sim['p_load'] / ss['p_load'] - 1):.2e}")
    print(f"  mechanical power {sim['p_mech']:.3f} W = load {sim['p_load']:.3f} + coil heat {sim['p_arm']:.3f} (+ stored change {sim['stored']:.1e} J)")
    print(f"  energy balance residual {energy_balance(sim):.2e} J of {sim['w_mech']:.4f} J")
    print(f"  mean torque {sim['torque'][-4 * 400:].mean() * 1000:.2f} mN m (negative = opposes rotation)")

    fig, (a1, a2) = plt.subplots(2, 1, figsize=(7, 6))
    tm = sim["t"]
    em_t = ss["em"] * np.sin(sim["omega"] * tm)
    a1.plot(tm[:1200] * 1000, em_t[:1200] / ss["z"], color="tab:gray", label="EMF / Z (no inductance lag)")
    a1.plot(tm[:1200] * 1000, sim["i"][:1200], color="tab:blue", label="RK4 current")
    ia = ss["i_peak"] * np.sin(sim["omega"] * tm - np.radians(ss["phase_deg"]))
    a1.plot(tm[:1200:15] * 1000, ia[:1200:15], "k.", ms=3, label="exact steady state")
    a1.set_xlabel("time (ms)")
    a1.set_ylabel("current (A)")
    a1.set_title("Loaded generator: start-up and steady state")
    a1.grid(alpha=0.3)
    a1.legend(fontsize=8)
    ln = slice(-1600, None)
    a2.plot((tm[ln] - tm[ln][0]) * 1000, sim["torque"][ln] * sim["omega"], color="tab:red", label="magnetic power on the coil = torque x w")
    a2.plot((tm[ln] - tm[ln][0]) * 1000, em_t[ln] * sim["i"][ln], color="tab:green", ls="--", label="e i (power we supply)")
    a2.axhline(0, color="k", lw=0.5)
    a2.set_xlabel("time over four periods (ms)")
    a2.set_ylabel("power (W)")
    a2.set_title("Lenz's law: the field takes power from the shaft")
    a2.grid(alpha=0.3)
    a2.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "generator_rk4.png", dpi=120)

    rls = np.logspace(np.log10(0.7), np.log10(500), 80)
    rows = [transformer_load(r) for r in rls]
    pout = np.array([r["p_out"] for r in rows])
    eff = np.array([r["eff"] for r in rows])
    reg = np.array([r["regulation"] for r in rows])
    rated = transformer_load(6.0)
    print(f"transformer at 6 ohm: V2 {rated['v2']:.2f} V, I2 {rated['i2']:.2f} A, P_out {rated['p_out']:.2f} W, "
          f"copper {rated['p_cu']:.2f} W, core {rated['p_core']:.2f} W, efficiency {rated['eff']:.1%}, regulation {rated['regulation']:.1%}")
    ib = int(np.argmax(eff))
    print(f"  best efficiency {eff[ib]:.1%} at P_out {pout[ib]:.1f} W; copper = core loss when I1 = {np.sqrt(rated['p_core'] / (XFMR['r1'] + (XFMR['n1'] / XFMR['n2']) ** 2 * XFMR['r2'])):.3f} A")
    fig2, (b1, b2) = plt.subplots(1, 2, figsize=(9, 4))
    b1.semilogx(pout, eff * 100, color="tab:blue")
    b1.plot(pout[ib], eff[ib] * 100, "ro")
    b1.set_xlabel("output power (W)")
    b1.set_ylabel("efficiency (%)")
    b1.set_title("Efficiency against load (illustrative)")
    b1.grid(alpha=0.3)
    b2.plot(pout, reg * 100, color="tab:red")
    b2.set_xlabel("output power (W)")
    b2.set_ylabel("voltage regulation (%)")
    b2.set_title("Output voltage drops as load rises")
    b2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "transformer_losses.png", dpi=120)

    fig3, ax = plt.subplots(1, 3, figsize=(10, 3.6), sharey=False)
    for axx, drive in zip(ax, (1.0, 2.0, 3.0)):
        r = saturating_core(drive)
        s0 = 5 * r["steps"]
        sl = slice(s0, s0 + r["steps"] + 1)
        tt = (r["t"][sl] - r["t"][s0]) * 1000
        axx.plot(tt, r["v_wind"][sl] * 24 / 460, color="tab:red", label="secondary voltage (V)")
        axx.set_xlabel("time (ms)")
        axx.set_title(f"drive {drive} x rated\nTHD {thd(r['v_wind'], r['steps'], r['periods']):.1%}", fontsize=9)
        axx.grid(alpha=0.3)
        ax2 = axx.twinx()
        ax2.plot(tt, r["i"][sl], color="tab:blue", lw=0.8)
        ax2.set_ylabel("magnetising current (A)", color="tab:blue", fontsize=8)
    ax[0].set_ylabel("secondary open-circuit voltage (V)")
    fig3.tight_layout()
    fig3.savefig(OUT / "core_saturation.png", dpi=120)
    for drive in (1.0, 2.0, 2.5, 3.0):
        r = saturating_core(drive)
        print(f"core drive {drive}: peak magnetising current {abs(r['i']).max():.3f} A, secondary THD {thd(r['v_wind'], r['steps'], r['periods']):.2%}")

    for n in (300, 310, 320, 330):
        print(f"dynamo N={n}: lamp power at 8/15/25 km/h = {dynamo_power(n, 8):.2f} / {dynamo_power(n, 15):.2f} / {dynamo_power(n, 25):.2f} W")

## Where this stops
The generator is a single coil with a fixed inductance, the transformer is a linear circuit
with made-up loss values, and the saturation curve is a tanh. They show the right trends and
let you test energy conservation, but they are not models for designing real machines.

## Coding tasks
1. **Easy.** Run the generator with RL = 5 ohm and then 20 ohm using `steady_state` and compare the load power with the 10 ohm case (13.56 W). Which gives more? Which wastes less in the armature?
2. **Stretch.** Use `saturating_core(2.2)` and `thd` to find the secondary THD at a drive of 2.2 times rated, and describe the waveform in your own words.
3. **Explain.** Explain why the energy balance must include the change in the energy stored in the inductor, and why that change is zero over whole periods.

## Assessment
- **A. Conceptual.** A bicycle dynamo is harder to pedal when the lamp is on than when the lamp is off. Explain why using Lenz's law, and say where the energy for the lamp comes from.
- **B. Numerical.** A generator has Em = 20 V peak, Ra = 1 ohm, RL = 9 ohm and negligible inductance. Find the RMS current, the power in the load, the heat in the armature and the mechanical power, and check that they balance.
- **C. Graph and simulation interpretation.** In the transformer efficiency figure, the output power is 0.29 W at a load of 500 ohm. Roughly what is the efficiency there (the curve starts near 16 percent), why is it so low, and why does it fall again beyond the peak at about 33 W?
- **D. Design and optimisation challenge.** Design: for the generator in the worked example (Ra = 2 ohm, L = 30 mH, 50 Hz, Em = 25.13 V), which RL gives the largest load power, and what share of the mechanical power does the armature waste there? What RL would you choose if the armature must waste no more than 10 percent, and what load power do you get?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    gen = dict(turns=250, b_field=0.2, area=0.0016, freq=50.0, ra=2.0, rl=10.0, ind=0.03)
    ss = L.steady_state(**gen)
    sim = L.simulate_generator(**gen)
    z_hand = np.sqrt(12.0 ** 2 + (2 * np.pi * 50 * 0.03) ** 2)
    checks.append((f"worked example: Z = 15.26 ohm, I_rms = 1.165 A, load power = 13.57 W (got {ss['z']:.2f}, {ss['i_rms']:.3f}, {ss['p_load']:.2f})",
                   abs(ss["z"] - z_hand) < 1e-9 and abs(ss["z"] - 15.26) < 0.01 and abs(ss["i_rms"] - 1.1647) < 1e-3 and abs(ss["p_load"] - 13.565) < 0.01))
    checks.append(("RK4 load power matches the exact steady state within 0.01 percent", abs(sim["p_load"] / ss["p_load"] - 1) < 1e-4))
    checks.append(("RK4 RMS current matches the exact steady state within 0.01 percent", abs(sim["i_rms"] / ss["i_rms"] - 1) < 1e-4))
    checks.append(("energy: mechanical work in = load energy + coil heat + change in stored energy",
                   abs(L.energy_balance(sim)) < 1e-6 * sim["w_mech"]))
    checks.append(("mechanical power = load power + coil heat power", abs(sim["p_mech"] - sim["p_load"] - sim["p_arm"]) < 1e-4))
    checks.append(("Lenz: the magnetic torque on the coil opposes the rotation on average", sim["torque"][-1600:].mean() < 0))
    k = 250 * 0.2 * 0.0016
    pw = sim["torque"] * sim["omega"]
    em = k * sim["omega"] * np.sin(sim["omega"] * sim["t"])
    checks.append(("magnetic power on the coil is exactly minus e times i at every instant",
                   float(np.abs(pw + em * sim["i"]).max()) < 1e-9))
    tiny = L.steady_state(250, 0.2, 0.0016, 50.0, 2.0, 10.0, 1e-9)
    hand = (tiny["em"] / 12.0) ** 2 / 2 * 10.0
    checks.append(("limit L -> 0: power is (Em / (Ra + RL))^2 / 2 * RL", abs(tiny["p_load"] - hand) < 1e-6))
    big = L.steady_state(250, 0.2, 0.0016, 50.0, 2.0, 10.0, 3.0)
    checks.append(("a large inductance cuts the current", big["i_rms"] < 0.1 * ss["i_rms"]))

    r = L.transformer_load(6.0)
    checks.append(("worked example: transformer at 6 ohm: V2 about 11.7 V, efficiency about 91.5 percent, regulation about 2.6 percent",
                   abs(r["v2"] - 11.68) < 0.02 and abs(r["eff"] - 0.915) < 0.002 and abs(r["regulation"] - 0.026) < 0.002))
    checks.append(("transformer power balance: P_in = P_out + copper + core",
                   abs(r["p_in"] - r["p_out"] - r["p_cu"] - r["p_core"]) < 1e-9))
    checks.append(("no load: output voltage is V1 / a and output power is tiny",
                   abs(L.transformer_load(1e9)["v2"] - 230.0 * 24 / 460) < 0.01 and L.transformer_load(1e9)["p_out"] < 1e-3))
    loads = np.logspace(np.log10(0.7), np.log10(500), 400)
    effs = np.array([L.transformer_load(x)["eff"] for x in loads])
    best = loads[int(np.argmax(effs))]
    rb = L.transformer_load(best)
    checks.append(("efficiency is best where copper loss is close to core loss", abs(rb["p_cu"] / rb["p_core"] - 1) < 0.1))
    regs = [L.transformer_load(x)["regulation"] for x in (50.0, 12.0, 6.0, 3.0)]
    checks.append(("regulation grows as the load grows", bool(np.all(np.diff(regs) > 0))))

    lo = L.saturating_core(0.5)
    hi = L.saturating_core(2.5)
    lam_pk = 0.5 * 12 * np.sqrt(2) / (2 * np.pi * 50)
    i_lin = lam_pk / (0.108 / 0.05)
    checks.append(("small drive: magnetising current matches the linear value lambda / L within 5 percent",
                   abs(abs(lo["i"][3 * lo["steps"]:]).max() / i_lin - 1) < 0.05))
    checks.append(("small drive: secondary voltage is a clean sine (THD below 0.1 percent)", L.thd(lo["v_wind"], lo["steps"], lo["periods"]) < 1e-3))
    checks.append(("saturation: current spikes more than 100 times and the output THD exceeds 10 percent",
                   abs(hi["i"][3 * hi["steps"]:]).max() > 100 * abs(lo["i"][3 * lo["steps"]:]).max() and L.thd(hi["v_wind"], hi["steps"], hi["periods"]) > 0.10))

    p15, p25 = L.dynamo_power(310, 15), L.dynamo_power(310, 25)
    checks.append((f"dynamo N = 310: at least 3 W at 15 km/h and at most 4.5 W at 25 km/h (got {p15:.2f} and {p25:.2f})", p15 >= 3.0 and p25 <= 4.5))
    checks.append(("dynamo N = 300 is too weak and N = 330 over-drives the lamp at 25 km/h",
                   L.dynamo_power(300, 15) < 3.0 and L.dynamo_power(330, 25) > 4.5))
    checks.append(("dynamo power grows more slowly than speed squared (self-limiting)",
                   L.dynamo_power(310, 25) / L.dynamo_power(310, 15) < (25 / 15) ** 2))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
